# Day 3: Segmentation, and an Honest Test of What Was Learned

The cup point cloud from Day 2 has exact labels (base / wall / lid), computed from
the parametric model rather than painted by hand. This notebook trains a small
PointNet-style segmentation head on it and then asks the only question this
dataset can actually answer.

## What this experiment can and cannot measure

**Cannot** measure generalisation to new shapes. There is one cup. Every sample
is the same 50 000 points resampled, so a random train/test split is not a
distribution shift at all. Any absolute IoU above ~0.9 is uninformative.

**Can** measure robustness to a pose change, because a pose change *is* a real
shift: the network sees coordinates it has never seen.

## The rotation that matters, and the one that does not

My first version of this experiment rotated the cup about its own vertical axis
(yaw) and tested at 30 degrees. The no-augmentation model kept mIoU 0.9946.

That was not the model being good. **It was the experiment being wrong.** The
labels are height bands, and yaw leaves the height coordinate untouched, so the
input-output relation is unchanged and there is nothing to fail at.

The rotation that actually breaks a height-band model is a **tilt** about a
horizontal axis, because that is what mixes $z$ into $x$ and $y$. This notebook
uses tilt.

## The four cells

|  | test 0 deg (upright) | test 30 deg tilt (unseen pose) |
| :--- | :--- | :--- |
| **train, upright only** | IoU should be high | IoU should collapse |
| **train, tilt augmentation** | IoU high | IoU should survive |

If the bottom-right cell does not survive, the augmentation did nothing. If the
top-right cell does not collapse, the task is solvable without using the pose,
and the honest conclusion is "this dataset cannot measure the effect" -- which is
also a result worth writing down.

## 0. Setup

In [ ]:
import json, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

BASE = Path(r"C:\Users\SHUS\Documents\deepseek-harness\default-workspace\Rankross-drafts\Cup_Practice")
OUT  = BASE / "model"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print("device:", device)
print("torch :", torch.__version__)

## 1. Load the point cloud

Check the labels before training on them. A silent label bug looks exactly like a
bad model.

In [ ]:
data   = np.load(OUT / "cup_points.npz", allow_pickle=True)
P      = data["points"].astype(np.float32)
LABELS = data["labels"].astype(np.int64)

K = 3
print("points :", P.shape)
print("labels :", np.bincount(LABELS, minlength=K), " (base / wall / lid)")
print("bbox   :", P.min(0).round(2), "->", P.max(0).round(2))

## 2. Dataset

Each sample is `N` points drawn without replacement from the same cloud, then
centred on its own centroid so that translation invariance is free and the pose
test is not contaminated by a translation bug.

The pose change is a **tilt about a horizontal axis**, optionally randomised in
both angle and azimuth during training. Angle is in degrees away from upright.

In [ ]:
def tilt_matrix(deg, azim_deg=0.0):
    """Rotation by `deg` degrees about a horizontal axis at azimuth `azim_deg`."""
    t = np.deg2rad(deg)
    a = np.deg2rad(azim_deg)
    axis = np.array([np.cos(a), np.sin(a), 0.0])
    Kx = np.array([[0.0, -axis[2], axis[1]],
                   [axis[2], 0.0, -axis[0]],
                   [-axis[1], axis[0], 0.0]])
    return np.eye(3) + np.sin(t) * Kx + (1.0 - np.cos(t)) * (Kx @ Kx)


class CupSegDataset(torch.utils.data.Dataset):
    def __init__(self, P, L, n=1024, n_samples=1200,
                 tilt_aug=0.0, fixed_tilt=None, jitter=0.0, seed=0):
        self.P, self.L = P, L
        self.n = n
        self.n_samples = n_samples
        self.tilt_aug = tilt_aug
        self.fixed_tilt = fixed_tilt
        self.jitter = jitter
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, i):
        idx = self.rng.choice(len(self.P), self.n, replace=False)
        x = self.P[idx].copy()
        y = self.L[idx]

        x -= x.mean(axis=0, keepdims=True)

        if self.tilt_aug > 0:
            R = tilt_matrix(self.rng.uniform(-self.tilt_aug, self.tilt_aug),
                            self.rng.uniform(0.0, 360.0))
            x = x @ R.T
        elif self.fixed_tilt is not None:
            R = tilt_matrix(*self.fixed_tilt)
            x = x @ R.T

        if self.jitter > 0:
            x += self.rng.normal(0.0, self.jitter, x.shape)

        return torch.from_numpy(x.astype(np.float32)), torch.from_numpy(y)


xb, yb = CupSegDataset(P, LABELS, n_samples=2)[0]
print("sample x:", tuple(xb.shape), " y:", tuple(yb.shape))
print("label counts in one sample:", torch.bincount(yb, minlength=K).tolist())

Pc = P - P.mean(0, keepdims=True)
cuts = np.array([Pc[:, 2].min() + 10.0, Pc[:, 2].min() + (213.0 - 32.0)])
for d in (0, 30):
    zr = (Pc @ tilt_matrix(d).T)[:, 2]
    acc = float((np.digitize(zr, cuts) == LABELS).mean())
    print(f"tilt {d:2d} deg -> z range {zr.min():8.2f} .. {zr.max():8.2f}"
          f"   z-only baseline acc {acc:.3f}")

This is the whole argument for using tilt, and it is why the first version of
this experiment was wrong.

Note that the **z range barely changes**: the cup is tall and thin, so tilting
trades a little height for a little width, and the extremes stay put. The range
is the misleading part. What changes is *which point sits at which height* -- a
point on the top rim can fall below the old rim threshold while a point near the
base rises above it.

The `z-only baseline` line measures exactly that. It assigns labels by absolute
height with thresholds set upright, so upright it is near 1.0, and at 30 degrees
it loses a large share of the points. That number is the ceiling of "coordinates
only", and it is the effect the four-cell table is really measuring.

## 3. Model

Per-point MLP -> global max pooling -> concatenate the global feature back onto
every point -> segmentation head. This is the segmentation half of PointNet.

`use_tnet` optionally adds a 3x3 input transform so the ablation can compare
*learning* an invariance against *augmenting* for it.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv1d(cin, cout, 1), nn.BatchNorm1d(cout), nn.ReLU(inplace=True),
    )


class TNet(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k
        self.conv = nn.Sequential(conv_block(k, 64), conv_block(64, 128), conv_block(128, 1024))
        self.fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Linear(256, k * k),
        )

    def forward(self, x):
        f = self.conv(x.transpose(1, 2)).max(dim=2).values
        A = self.fc(f).view(-1, self.k, self.k)
        return A + torch.eye(self.k, device=x.device).unsqueeze(0)


class MiniPointNetSeg(nn.Module):
    def __init__(self, k=3, n_cls=3, use_tnet=False):
        super().__init__()
        self.tnet = TNet(k) if use_tnet else None
        self.mlp = nn.Sequential(conv_block(k, 64), conv_block(64, 128), conv_block(128, 1024))
        self.head = nn.Sequential(
            conv_block(1024 + 1024, 512), conv_block(512, 256), nn.Conv1d(256, n_cls, 1),
        )

    def forward(self, x):
        B, N, _ = x.shape
        reg = torch.zeros((), device=x.device)
        if self.tnet is not None:
            A = self.tnet(x)
            x = torch.bmm(x, A.transpose(1, 2))
            I = torch.eye(A.size(1), device=A.device).unsqueeze(0)
            reg = ((I - torch.bmm(A, A.transpose(1, 2))) ** 2).sum(dim=(1, 2)).mean()

        f = self.mlp(x.transpose(1, 2))
        g = f.max(dim=2, keepdim=True).values
        h = torch.cat([f, g.expand(-1, -1, N)], dim=1)
        return self.head(h).transpose(1, 2), reg


m = MiniPointNetSeg().to(device)
with torch.no_grad():
    out, _ = m(torch.randn(2, 64, 3, device=device))
print("output shape:", tuple(out.shape), " expected (B, N, 3)")

## 4. Train / evaluate helpers

In [ ]:
def train_model(tilt_aug=0.0, epochs=15, n_samples=1200, n=1024, bs=16,
                lr=1e-3, use_tnet=False, jitter=0.01, verbose=False):
    ds = CupSegDataset(P, LABELS, n=n, n_samples=n_samples,
                       tilt_aug=tilt_aug, jitter=jitter, seed=1)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True)

    torch.manual_seed(0); np.random.seed(0)
    model = MiniPointNetSeg(use_tnet=use_tnet).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []

    for ep in range(1, epochs + 1):
        model.train()
        tot, cnt = 0.0, 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            logits, reg = model(x)
            loss = F.cross_entropy(logits.reshape(-1, K), y.reshape(-1))
            if use_tnet:
                loss = loss + 0.001 * reg
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach().item() * len(x); cnt += len(x)
        hist.append(tot / max(cnt, 1))
        if verbose and (ep % 5 == 0 or ep == 1):
            print(f"  epoch {ep:3d}  loss {hist[-1]:.4f}")
    return model, hist


@torch.no_grad()
def evaluate(model, fixed_tilt, n_eval=120, n=1024, bs=24):
    ds = CupSegDataset(P, LABELS, n=n, n_samples=n_eval,
                       fixed_tilt=fixed_tilt, seed=99)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False)
    model.eval()
    conf = np.zeros((K, K), dtype=np.int64)
    for x, y in dl:
        logits, _ = model(x.to(device))
        pred = logits.argmax(dim=-1).cpu().numpy().reshape(-1)
        np.add.at(conf, (y.numpy().reshape(-1), pred), 1)

    iou = []
    for c in range(K):
        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp
        den = tp + fp + fn
        iou.append(float(tp / den) if den > 0 else float("nan"))
    return float(np.nanmean(iou)), iou, float(np.trace(conf) / conf.sum()), conf

## 5. The four-cell ablation

Two models, identical except for augmentation. Each is evaluated upright and at
a 30 degree tilt it has never seen.

In [ ]:
CELLS = {}
CONFIGS = [(0.0, "upright only"), (30.0, "tilt aug +/-30")]
TESTS = [(0, (0.0, 0.0)), (30, (30.0, 0.0))]

for aug, label in CONFIGS:
    print(f"training: {label}")
    t0 = time.time()
    model, hist = train_model(tilt_aug=aug, epochs=15, verbose=True)
    for rot_deg, tilt in TESTS:
        miou, per, acc, conf = evaluate(model, fixed_tilt=tilt)
        CELLS[(aug, rot_deg)] = {"mIoU": miou, "per_class": per, "acc": acc, "conf": conf}
        print(f"   test {rot_deg:2d} deg -> mIoU {miou:.4f}  acc {acc:.4f}")
    print(f"   ({time.time()-t0:.1f}s)\n")

In [ ]:
names = ["base", "wall", "lid"]
head = "train / test"
print("mIoU table")
print("-" * 52)
print(f"{head:20s} {'0 deg':>10s} {'30 deg':>10s} {'drop':>10s}")
for aug, label in CONFIGS:
    a = CELLS[(aug, 0)]["mIoU"]
    b = CELLS[(aug, 30)]["mIoU"]
    drop = 100.0 * (a - b) / a if a else float("nan")
    print(f"{label:20s} {a:10.4f} {b:10.4f} {drop:+9.1f}%")

print()
print("per-class IoU")
print("-" * 78)
print(f"{head:20s}" + "".join(f"{n:>19s}" for n in names))
for aug, label in CONFIGS:
    for rot_deg in (0, 30):
        per = CELLS[(aug, rot_deg)]["per_class"]
        row = "".join(f"{v:19.3f}" for v in per)
        tag = label + " @" + str(rot_deg) + "d"
        print(f"{tag:20s}{row}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
panels = [
    ((0.0, 0),  "upright / test 0"),
    ((0.0, 30), "upright / test 30"),
    ((30.0, 0), "aug / test 0"),
    ((30.0, 30),"aug / test 30"),
]
for ax, (key, title) in zip(axes, panels):
    conf = CELLS[key]["conf"]
    disp = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)
    ax.imshow(disp, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_xticklabels(names)
    ax.set_yticks(range(K)); ax.set_yticklabels(names)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"{title}\nmIoU {CELLS[key]['mIoU']:.3f}", fontsize=9)
plt.tight_layout(); plt.show()

## 6. Three unit tests

Each catches a specific bug class. Note what test 1 does **not** prove.

In [ ]:
def unit_tests():
    torch.manual_seed(0)
    model = MiniPointNetSeg().to(device).eval()
    x = torch.randn(4, 512, 3, device=device)

    with torch.no_grad():
        a, _ = model(x)
        perm = torch.randperm(x.size(1), device=device)
        b, _ = model(x[:, perm, :])
    d1 = float((a - b[:, torch.argsort(perm), :]).abs().max())

    with torch.no_grad():
        solo, _ = model(x[:1])
        batched, _ = model(x)
    d2 = float((solo - batched[:1]).abs().max())

    with torch.no_grad():
        out, _ = model(x)
    shape_ok = tuple(out.shape) == (4, 512, 3)

    print(f"permutation invariance, max|delta| : {d1:.3e}   {'PASS' if d1 < 1e-5 else 'FAIL'}")
    print(f"batch coupling,        max|delta| : {d2:.3e}   "
          f"{'PASS' if d2 < 1e-4 else 'FAIL'}   (1e-4: GPU kernels reorder reductions)")
    print(f"output shape (4,512,3)            : {tuple(out.shape)}   {'PASS' if shape_ok else 'FAIL'}")
    return d1, d2, shape_ok


unit_tests()

### What test 1 is really telling you

It passes on a freshly initialised network, because
$\gamma \circ \text{MAX} \circ h$ is invariant for any $h$. A passing permutation
test is evidence about the *architecture's* symmetry, never about learning.

The evidence about learning is the four-cell table: that is a number a randomly
initialised network cannot fake.

### Optional: augmentation versus T-Net

Both are ways to buy pose robustness. Run the cell below to put a third
configuration in the table: no augmentation, but with a learned 3x3 input
transform.

In [ ]:
model_t, _ = train_model(tilt_aug=0.0, epochs=15, use_tnet=True, verbose=True)
for rot_deg, tilt in TESTS:
    miou, per, acc, conf = evaluate(model_t, fixed_tilt=tilt)
    CELLS[("tnet", rot_deg)] = {"mIoU": miou, "per_class": per, "acc": acc, "conf": conf}
    print(f"   T-Net, test {rot_deg:2d} deg -> mIoU {miou:.4f}")

a = CELLS[("tnet", 0)]["mIoU"]; b = CELLS[("tnet", 30)]["mIoU"]
print(f"   T-Net drop: {100.0*(a-b)/a:+.1f}%")

## 7. Results

Fill in after running, then copy into `DAY3_LOG.md`.

| Cell | mIoU | Notes |
| :--- | ---: | :--- |
| train upright, test 0 | | seen pose |
| train upright, test 30 | | unseen pose, expected to drop |
| train tilt-aug, test 0 | | |
| train tilt-aug, test 30 | | unseen pose, expected to hold |
| (optional) T-Net, test 30 | | learned invariance instead of augmentation |

### What I will write in the log

- The absolute IoU is uninformative here (one shape, exact labels, easy bands).
  The number that carries information is the **drop** between columns.
- **The collapsed cell is not reproducible to two digits.** Two runs with the
  same seed gave mIoU 0.25 and 0.37 at 30 degrees, because CUDA convolution is
  not bit-deterministic. The conclusion (it collapses) is robust; the value is
  not. Report a range, or average three seeds, and say so.
- The first version of this experiment used yaw and showed no drop. That was the
  experiment being wrong, not the model being robust. Recorded because it is the
  more useful lesson.
- If the bottom-right cell does not survive either, the conclusion is "tilt
  augmentation was insufficient at this budget", not "the model generalises".